# Helmet Compliance Detector — Colab Training Notebook

Trains both the YOLOv8 baseline and the YOLOv8+CBAM variant on a free-tier Colab GPU (T4).

This notebook runs **entirely on Colab's local disk** — code cloned from GitHub, dataset
downloaded via `gdown`, checkpoints saved locally — deliberately avoiding the Drive
*mount* (`/content/drive`) for anything performance- or reliability-sensitive. We hit
real, repeated problems with it (large files not visible to the mount even after
`force_remount`, and eventually the mount crashing entirely mid-session with
`OSError: Transport endpoint is not connected`), so the whole pipeline was restructured
to not depend on it. You'll download the two final `best.pt` files at the end straight
from Colab's own file browser — no Drive round-trip needed.

Steps: get project code -> install deps -> download dataset -> train baseline -> train
CBAM -> compare -> download results.</cell id="cell-0">

In [ ]:
!nvidia-smi

In [ ]:
CODE_DIR = "/content/od"  # local disk -- see the intro cell for why this isn't on Drive

## Get the project code

Cloned straight from GitHub (public repo, no credentials needed) to local disk. Re-running
this cell later in the *same* runtime detects the existing clone and `git pull`s instead
of trying to clone again; a fresh runtime just clones fresh (cheap, it's a small repo).</cell id="cell-3">

In [ ]:
import os
if os.path.isdir(f"{CODE_DIR}/.git"):
    print("Repo already cloned, pulling latest instead...")
    !cd {CODE_DIR} && git pull
else:
    !git clone https://github.com/rozengoza/od.git {CODE_DIR}
%cd {CODE_DIR}
!pip install -q -r requirements.txt

## Get the prepared dataset

`data_dataset.zip` (created locally by `python -m zipfile -c data_dataset.zip
data/dataset`) needs to be in your Google Drive, **link-shared ("Anyone with the
link")** — get its file ID from Drive's share link
(`.../file/d/<FILE_ID>/view?...`) and set `DATASET_ZIP_FILE_ID` below.

Downloaded via `gdown` (direct Drive-API download by file ID) straight to `/content/data`
— Colab's local disk. This intentionally never touches the `/content/drive` mount at all
(see the intro cell) — `gdown` only needs the file to be link-shared, not the mount to be
healthy.</cell id="cell-5">

In [ ]:
import os

# See markdown above for why this is a direct-by-ID download rather than a Drive-mount
# read. Get this ID from your own data_dataset.zip's share link if it differs.
DATASET_ZIP_FILE_ID = "1kgEeGRvWdG2avNRd0ywh_hROnkUmW1lE"
LOCAL_DATA_DIR = "/content/data"
zip_path = "/content/data_dataset.zip"

import gdown
gdown.download(id=DATASET_ZIP_FILE_ID, output=zip_path, quiet=False)

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
# NOTE: the zip's top-level entry is "dataset/" (python -m zipfile -c only keeps the
# basename of the source path), so extracting into LOCAL_DATA_DIR lands it at
# LOCAL_DATA_DIR/dataset/.
!python -m zipfile -e {zip_path} {LOCAL_DATA_DIR}

# The zip may still contain an older data.yaml with a machine-specific absolute "path:"
# line baked in (from whatever machine originally ran prepare_dataset.py) — strip it so
# Ultralytics falls back to resolving train/val/test relative to this yaml's own folder.
yaml_path = f"{LOCAL_DATA_DIR}/dataset/data.yaml"
lines = [l for l in open(yaml_path).read().splitlines() if not l.startswith("path:")]
open(yaml_path, "w").write("\n".join(lines) + "\n")

DATA_YAML = yaml_path  # used by every training/eval cell below
!ls {LOCAL_DATA_DIR}/dataset && cat {DATA_YAML}

### (Alternative) download fresh from Kaggle instead

Only needed if you didn't prepare the dataset locally. Uncomment and run instead of the
unzip cell above.

In [ ]:
# from google.colab import files
# import os
# os.makedirs('/root/.kaggle', exist_ok=True)
# uploaded = files.upload()  # select kaggle.json
# for fname in uploaded:
#     os.rename(fname, '/root/.kaggle/kaggle.json')
# os.chmod('/root/.kaggle/kaggle.json', 0o600)
# !python data/prepare_dataset.py --out data/dataset --val-frac 0.1 --test-frac 0.1

## Train baseline YOLOv8s

In [ ]:
!python train.py --variant baseline --data {DATA_YAML} --model-size s --epochs 60 --imgsz 640 --batch 16

**Before starting CBAM training below**: go download this baseline `best.pt` right now
via the file browser (see the "After training" section near the end of this notebook for
exact steps) rather than waiting until both runs are done. CBAM takes ~2 more hours;
better to have the baseline result safely off Colab in case anything interrupts the
session again.

## Train YOLOv8s + CBAM (novel-method variant)

In [ ]:
!python train.py --variant cbam --data {DATA_YAML} --model-size s --epochs 60 --imgsz 640 --batch 16

## Compare baseline vs CBAM on the held-out test split

In [ ]:
import glob, os

def latest_best(pattern):
    candidates = glob.glob(pattern)
    if not candidates:
        raise FileNotFoundError(f"No weights found matching {pattern}")
    return max(candidates, key=os.path.getmtime)

# Ultralytics auto-increments run folder names (helmet-baseline-yolov8s, ...s2, ...s3, ...)
# whenever a folder from an earlier attempt already exists, so don't assume the unsuffixed
# name is the real one — glob for it and take the most recently modified match instead.
RUNS_DIR = f"{CODE_DIR}/runs/detect"
baseline_weights = latest_best(f"{RUNS_DIR}/helmet-baseline-yolov8s*/weights/best.pt")
cbam_weights = latest_best(f"{RUNS_DIR}/helmet-cbam-yolov8s*/weights/best.pt")
print("baseline weights:", baseline_weights)
print("cbam weights:", cbam_weights)

!python evaluate.py \
  --weights {baseline_weights} {cbam_weights} \
  --names baseline cbam \
  --data {DATA_YAML} \
  --out docs/results_comparison.csv

## (Optional) Quick sanity check on a sample image

In [ ]:
import glob, os
from ultralytics import YOLO
from models import register_modules  # only needed for the CBAM checkpoint

cbam_weights = max(
    glob.glob(f"{CODE_DIR}/runs/detect/helmet-cbam-yolov8s*/weights/best.pt"),
    key=os.path.getmtime,
)
print("Using:", cbam_weights)
model = YOLO(cbam_weights)
results = model.predict(f"{LOCAL_DATA_DIR}/dataset/test/images", save=True, conf=0.35)
print('Annotated predictions saved under runs/detect/predict*/')

## After training: bring weights back to your local machine

Checkpoints now live on **local disk** (`/content/od/runs/detect/...`), not Drive.
**Check the actual folder names first** — Ultralytics auto-increments them
(`helmet-baseline-yolov8s`, `...s2`, `...s3`, ...) whenever a folder from an earlier
attempt already exists, so don't assume the unsuffixed name is right; the comparison
cell above already printed the exact paths it used (`baseline weights: ...`,
`cbam weights: ...`) — those are the ones to grab.

**Download directly via Colab's file browser** (left sidebar, folder icon): navigate to
`od/runs/detect/<run-name>/weights/`, right-click `best.pt` -> **Download**. Do this for
both the baseline and CBAM runs. No Drive involved at all.

Place each downloaded file locally as:
```
runs/detect/helmet-baseline-yolov8s/weights/best.pt
runs/detect/helmet-cbam-yolov8s/weights/best.pt
```
(drop any numeric suffix from the folder name when you copy it locally — that's the
plain name `evaluate.py` and `app/streamlit_app.py` expect by default).

**Do this promptly** — `/content` is wiped when the runtime disconnects or recycles, so
don't leave the downloads for "later" in a separate session.</cell id="cell-17">